# Chapter 10 Computational Lab
## Important Continuous Distributions

This notebook accompanies Chapter 10 of *Probability Theory with Python and AI*.

Chapter 9 developed general distributions with densities. This chapter focuses on three families that recur throughout probability:

$$
\operatorname{Exp}(\lambda),
\qquad
\operatorname{Gamma}(\alpha,\lambda),
\qquad
N(\mu,\sigma^2).
$$

The emphasis is not on memorizing formulas, but on understanding normalization, cdfs, moments, parameter effects, structural properties and the links among the three families.

### Learning goals

By the end of the lab you should be able to:

1. derive the cdf and survival function of an exponential variable;
2. compute exponential moments and use memorylessness correctly;
3. understand why memorylessness characterizes the exponential law under the stated positivity assumption;
4. simulate exponential variables by inverse transformation;
5. use the gamma function and its recurrence;
6. work consistently with the **shape--rate** gamma parameterization;
7. distinguish the three gamma shape regimes $0<\alpha<1$, $\alpha=1$, and $\alpha>1$;
8. compute gamma moments and identify the interior mode when it exists;
9. understand the Gaussian-integral normalization argument;
10. use the standard normal cdf $\Phi$ and its symmetry;
11. standardize a general normal variable;
12. distinguish the roles of $\mu$, $\sigma$, and $\sigma^2$;
13. compute normal interval probabilities;
14. compare exponential, gamma and normal laws structurally;
15. use the de Moivre--Laplace normal approximation with continuity correction;
16. distinguish numerical approximation from a limiting theorem;
17. audit AI-generated claims about these continuous families.

> **Expectation convention.** All expectation formulas here are density representations of the general expectation already constructed in Chapter 7.


## 0. Setup

The notebook implements the main formulas directly with Python's standard library, NumPy and Matplotlib. It does not rely on black-box distribution objects for the central derivations.


In [ ]:
from math import comb, erf, exp, factorial, gamma, lgamma, log, pi, sqrt
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def numerical_integral(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)


def exp_density(x, lam):
    x = np.asarray(x, dtype=float)
    return np.where(x >= 0, lam*np.exp(-lam*x), 0.0)


def exp_cdf(x, lam):
    x = np.asarray(x, dtype=float)
    return np.where(x < 0, 0.0, 1-np.exp(-lam*x))


def gamma_density(x, alpha, lam):
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)

    mask = x > 0
    if np.any(mask):
        logf = (
            alpha*math.log(lam)
            - math.lgamma(alpha)
            + (alpha-1)*np.log(x[mask])
            - lam*x[mask]
        )
        out[mask] = np.exp(logf)

    return out


def gamma_cdf_numerical(x, alpha, lam, n_grid=10000):
    if x <= 0:
        return 0.0

    # Avoid evaluating exactly at zero when alpha<1.
    eps = min(1e-8, x/100000)
    grid = np.linspace(eps, x, n_grid)
    f = gamma_density(grid, alpha, lam)

    # Add a small near-zero contribution numerically through a finer local grid.
    if alpha < 1:
        local = np.geomspace(max(1e-14, eps*1e-6), eps, 1500)
        local_f = gamma_density(local, alpha, lam)
        near_zero = numerical_integral(local_f, local)
    else:
        near_zero = 0.0

    return float(near_zero + numerical_integral(f, grid))


def phi(z):
    z = np.asarray(z, dtype=float)
    return np.exp(-z*z/2)/math.sqrt(2*math.pi)


def Phi_scalar(z):
    return 0.5*(1 + math.erf(z/math.sqrt(2)))


def Phi(z):
    z = np.asarray(z, dtype=float)
    return np.vectorize(Phi_scalar)(z)


def normal_density(x, mu, sigma):
    x = np.asarray(x, dtype=float)
    return np.exp(-((x-mu)**2)/(2*sigma*sigma))/(sigma*math.sqrt(2*math.pi))


def normal_cdf(x, mu, sigma):
    x = np.asarray(x, dtype=float)
    return Phi((x-mu)/sigma)


def binomial_pmf(k, n, p):
    if not isinstance(k, int) or k < 0 or k > n:
        return 0.0
    return comb(n, k)*(p**k)*((1-p)**(n-k))


def binomial_interval_probability(a, b, n, p):
    return sum(binomial_pmf(k, n, p) for k in range(a, b+1))


def normal_approx_binomial_interval(a, b, n, p, continuity=True):
    mu = n*p
    sigma = math.sqrt(n*p*(1-p))

    if continuity:
        lower = a - 0.5
        upper = b + 0.5
    else:
        lower = a
        upper = b

    return (
        Phi_scalar((upper-mu)/sigma)
        - Phi_scalar((lower-mu)/sigma)
    )


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Continuous-family tools are ready."
    "</div>"
))


## 1. The exponential distribution

For $\lambda>0$,

$$
X\sim\operatorname{Exp}(\lambda)
$$

means that

$$
f_X(x)
=
\begin{cases}
\lambda e^{-\lambda x},&x\ge0,\\
0,&x<0.
\end{cases}
$$

The parameter $\lambda$ is a **rate**. Larger rates concentrate more probability near zero.


### Normalization

The proposed density must first be checked:

$$
\int_{-\infty}^{\infty}f_X(x)\,dx
=
\int_0^\infty
\lambda e^{-\lambda x}\,dx
=
1.
$$


In [ ]:
exp_lam_norm = widgets.FloatSlider(
    value=2.0, min=0.2, max=5.0, step=0.1, description="lambda"
)
exp_norm_output = widgets.Output()


def update_exp_normalization(*_):
    with exp_norm_output:
        clear_output(wait=True)

        lam = exp_lam_norm.value
        upper = 20/max(lam, 0.2)
        grid = np.linspace(0, upper, 20000)
        mass = numerical_integral(exp_density(grid, lam), grid)

        display(Math(
            r"\int_0^\infty \lambda e^{-\lambda x}\,dx=1"
        ))
        display(Math(
            r"\text{numerical truncated check}\approx" + f"{mass:.10f}"
        ))


exp_lam_norm.observe(update_exp_normalization, names="value")
display(widgets.VBox([exp_lam_norm, exp_norm_output]))
update_exp_normalization()


### Parameter effect: changing the rate

Increasing $\lambda$:

- increases the density height near $0$;
- decreases the mean $1/\lambda$;
- decreases the variance $1/\lambda^2$;
- makes the right tail decay faster.


In [ ]:
x = np.linspace(0, 6, 800)

fig, ax = plt.subplots(figsize=(8, 3.6))
for lam in [0.5, 1.0, 2.0]:
    ax.plot(x, exp_density(x, lam), label=f"lambda={lam:g}")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Exponential densities for different rates")
ax.legend()
plt.show()


### Cdf and survival function

For $x\ge0$,

$$
F_X(x)
=
1-e^{-\lambda x},
$$

and

$$
\boxed{
P(X>x)
=
e^{-\lambda x}.
}
$$

For $0\le a<b$,

$$
P(a<X\le b)
=
e^{-\lambda a}
-
e^{-\lambda b}.
$$


In [ ]:
exp_lam = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description="lambda")
exp_a = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="a")
exp_b = widgets.FloatSlider(value=2.0, min=0.1, max=7.0, step=0.1, description="b")
exp_prob_output = widgets.Output()


def update_exp_prob(*_):
    with exp_prob_output:
        clear_output(wait=True)

        lam = exp_lam.value
        a = exp_a.value
        b = exp_b.value

        if not a < b:
            display(Markdown("**Require a<b.**"))
            return

        tail = math.exp(-lam*b)
        interval = math.exp(-lam*a) - math.exp(-lam*b)

        display(Math(
            r"P(X>" + f"{b:g}" + r")=" + f"{tail:.8f}"
        ))
        display(Math(
            r"P(" + f"{a:g}" + r"<X\le" + f"{b:g}" + r")="
            + f"{interval:.8f}"
        ))


for control in (exp_lam, exp_a, exp_b):
    control.observe(update_exp_prob, names="value")

display(widgets.VBox([
    widgets.HBox([exp_lam, exp_a, exp_b]),
    exp_prob_output,
]))
update_exp_prob()


### Mean and variance

If

$$
X\sim\operatorname{Exp}(\lambda),
$$

then

$$
\boxed{
\mathbb E[X]
=
\frac1\lambda,
\qquad
\operatorname{Var}(X)
=
\frac1{\lambda^2}.
}
$$

The density integral is a computational representation of the general expectation from Chapter 7.


In [ ]:
rate_from_mean = widgets.FloatSlider(
    value=5.0, min=0.5, max=20.0, step=0.5, description="mean"
)
rate_output = widgets.Output()


def update_rate_from_mean(*_):
    with rate_output:
        clear_output(wait=True)

        m = rate_from_mean.value
        lam = 1/m
        variance = 1/(lam*lam)

        display(Math(r"\lambda=" + f"{lam:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))


rate_from_mean.observe(update_rate_from_mean, names="value")
display(widgets.VBox([rate_from_mean, rate_output]))
update_rate_from_mean()


## 2. Memorylessness

The exponential law satisfies

$$
\boxed{
P(X>s+t\mid X>s)
=
P(X>t),
\qquad
s,t\ge0.
}
$$

This is an exact identity of the model, not a large-time approximation.


In [ ]:
mem_lam = widgets.FloatSlider(value=0.2, min=0.05, max=2.0, step=0.05, description="lambda")
mem_s = widgets.FloatSlider(value=5.0, min=0, max=20, step=0.5, description="s")
mem_t = widgets.FloatSlider(value=3.0, min=0, max=20, step=0.5, description="t")
mem_output = widgets.Output()


def update_memoryless(*_):
    with mem_output:
        clear_output(wait=True)

        lam = mem_lam.value
        s = mem_s.value
        t = mem_t.value

        lhs = math.exp(-lam*(s+t))/math.exp(-lam*s)
        rhs = math.exp(-lam*t)

        display(Math(
            r"P(X>s+t\mid X>s)=" + f"{lhs:.8f}"
        ))
        display(Math(
            r"P(X>t)=" + f"{rhs:.8f}"
        ))


for control in (mem_lam, mem_s, mem_t):
    control.observe(update_memoryless, names="value")

display(widgets.VBox([
    widgets.HBox([mem_lam, mem_s, mem_t]),
    mem_output,
]))
update_memoryless()


### A non-memoryless continuous law

If

$$
Y\sim U(0,10),
$$

then

$$
P(Y>8\mid Y>5)
=
\frac25,
$$

but

$$
P(Y>3)
=
\frac7{10}.
$$

Thus memorylessness is a restrictive structural property, not a generic consequence of having a density.


In [ ]:
display(Math(r"P(Y>8\mid Y>5)=0.4"))
display(Math(r"P(Y>3)=0.7"))


### Characterization of the exponential law

Let

$$
S(t)=P(X>t)>0.
$$

If

$$
S(s+t)=S(s)S(t)
$$

for all $s,t\ge0$, define

$$
g(t)=-\log S(t).
$$

Then

$$
g(s+t)=g(s)+g(t).
$$

Because $S$ is non-increasing, $g$ is non-decreasing. Additivity plus monotonicity forces

$$
g(t)=\lambda t,
$$

so

$$
S(t)=e^{-\lambda t}
$$

for a unique $\lambda>0$.


In [ ]:
char_lam = widgets.FloatSlider(value=0.7, min=0.1, max=2.0, step=0.1, description="lambda")
char_output = widgets.Output()


def update_characterization(*_):
    with char_output:
        clear_output(wait=True)

        lam = char_lam.value
        t = np.linspace(0, 6, 500)
        S = np.exp(-lam*t)
        g = -np.log(S)

        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.plot(t, g)
        ax.set_xlabel("t")
        ax.set_ylabel("-log S(t)")
        ax.set_title("Exponential survival becomes a linear additive function")
        plt.show()


char_lam.observe(update_characterization, names="value")
display(widgets.VBox([char_lam, char_output]))
update_characterization()


## 3. Inverse construction of an exponential variable

If

$$
U\sim U(0,1),
$$

then

$$
\boxed{
X
=
-\frac1\lambda\log(1-U)
}
$$

has the $\operatorname{Exp}(\lambda)$ distribution, apart from an arbitrary assignment at the null endpoint $U=1$.

Since $1-U\sim U(0,1)$, the equivalent formula

$$
-\frac1\lambda\log U
$$

has the same distribution.


In [ ]:
inv_lam = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description="lambda")
inv_u = widgets.FloatSlider(value=0.8, min=0.001, max=0.999, step=0.001, description="U")
inv_output = widgets.Output()


def update_inverse_exp(*_):
    with inv_output:
        clear_output(wait=True)

        lam = inv_lam.value
        u = inv_u.value
        x = -math.log(1-u)/lam

        display(Math(
            r"X=-\frac1\lambda\log(1-U)=" + f"{x:.8f}"
        ))


for control in (inv_lam, inv_u):
    control.observe(update_inverse_exp, names="value")

display(widgets.VBox([
    widgets.HBox([inv_lam, inv_u]),
    inv_output,
]))
update_inverse_exp()


In [ ]:
inv_N = widgets.IntSlider(value=10000, min=500, max=50000, step=500, description="N")
inv_sim_lam = widgets.FloatSlider(value=2.0, min=0.2, max=5.0, step=0.1, description="lambda")
inv_sim_output = widgets.Output()


def update_inverse_sim(*_):
    with inv_sim_output:
        clear_output(wait=True)

        N = inv_N.value
        lam = inv_sim_lam.value

        rng = np.random.default_rng(2026)
        U = rng.random(N)
        X = -np.log(1-U)/lam

        display(Math(r"\text{sample mean}\approx" + f"{X.mean():.6f}"))
        display(Math(r"\text{theoretical mean}=" + f"{1/lam:.6f}"))

        grid = np.linspace(0, np.quantile(X, 0.995), 500)

        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.hist(X, bins=50, density=True)
        ax.plot(grid, exp_density(grid, lam))
        ax.set_xlabel("x")
        ax.set_ylabel("density")
        ax.set_title("Inverse-transform exponential simulation")
        plt.show()


for control in (inv_N, inv_sim_lam):
    control.observe(update_inverse_sim, names="value")

display(widgets.VBox([
    widgets.HBox([inv_N, inv_sim_lam]),
    inv_sim_output,
]))
update_inverse_sim()


## 4. The gamma function

For $\alpha>0$,

$$
\boxed{
\Gamma(\alpha)
=
\int_0^\infty
x^{\alpha-1}e^{-x}\,dx.
}
$$

The integral is improper at infinity. For $0<\alpha<1$, it also has an integrable singularity at the origin.


### Gamma recurrence

Integration by parts gives

$$
\boxed{
\Gamma(\alpha+1)
=
\alpha\Gamma(\alpha).
}
$$

Since

$$
\Gamma(1)=1,
$$

for positive integers $n$,

$$
\boxed{
\Gamma(n)
=
(n-1)!.
}
$$

The shift by one is important.


In [ ]:
gamma_n = widgets.IntSlider(value=4, min=1, max=12, description="n")
gamma_rec_output = widgets.Output()


def update_gamma_recurrence(*_):
    with gamma_rec_output:
        clear_output(wait=True)

        n = gamma_n.value
        display(Math(
            r"\Gamma(" + str(n) + r")=" + f"{math.gamma(n):g}"
        ))
        display(Math(
            r"(" + str(n-1) + r")!=" + f"{math.factorial(n-1):g}"
        ))


gamma_n.observe(update_gamma_recurrence, names="value")
display(widgets.VBox([gamma_n, gamma_rec_output]))
update_gamma_recurrence()


### Special value

The Gaussian integral will imply

$$
\boxed{
\Gamma\left(\frac12\right)
=
\sqrt\pi.
}
$$

Then the recurrence gives

$$
\Gamma\left(\frac32\right)
=
\frac{\sqrt\pi}{2},
$$

and

$$
\Gamma\left(\frac52\right)
=
\frac{3\sqrt\pi}{4}.
$$


## 5. Gamma distribution: shape--rate form

For $\alpha>0$ and $\lambda>0$,

$$
X\sim\operatorname{Gamma}(\alpha,\lambda)
$$

means

$$
f_X(x)
=
\begin{cases}
\dfrac{\lambda^\alpha}{\Gamma(\alpha)}
x^{\alpha-1}e^{-\lambda x},
&x>0,\\
0,&x\le0.
\end{cases}
$$

This book uses the **rate** $\lambda$, not the scale $\theta=1/\lambda$.


### Density convention near zero

For $\alpha\ge1$, the displayed gamma density fits the Riemann-density framework used earlier.

For

$$
0<\alpha<1,
$$

the factor

$$
x^{\alpha-1}
$$

diverges as $x\downarrow0$, but the singularity is integrable. In that regime the density is interpreted as a Lebesgue density, equivalently through its non-negative improper integrals.


In [ ]:
gamma_alpha_norm = widgets.FloatSlider(value=3.0, min=0.3, max=6.0, step=0.1, description="alpha")
gamma_lam_norm = widgets.FloatSlider(value=2.0, min=0.2, max=5.0, step=0.1, description="lambda")
gamma_norm_output = widgets.Output()


def update_gamma_normalization(*_):
    with gamma_norm_output:
        clear_output(wait=True)

        alpha = gamma_alpha_norm.value
        lam = gamma_lam_norm.value

        # Numerical mass over a wide finite range, avoiding exact zero.
        upper = max(20/lam, alpha/lam + 12*math.sqrt(alpha)/lam)
        grid = np.geomspace(1e-8, upper, 40000)
        mass = numerical_integral(gamma_density(grid, alpha, lam), grid)

        display(Math(
            r"\text{numerical total mass}\approx" + f"{mass:.8f}"
        ))


for control in (gamma_alpha_norm, gamma_lam_norm):
    control.observe(update_gamma_normalization, names="value")

display(widgets.VBox([
    widgets.HBox([gamma_alpha_norm, gamma_lam_norm]),
    gamma_norm_output,
]))
update_gamma_normalization()


### Gamma cdf

For $x>0$,

$$
F_X(x)
=
\frac1{\Gamma(\alpha)}
\int_0^{\lambda x}
u^{\alpha-1}e^{-u}\,du.
$$

The numerator is the lower incomplete gamma function.

For general $\alpha$, the cdf does not have an elementary closed form.


### Integer-shape example

If

$$
X\sim\operatorname{Gamma}(2,1),
$$

then

$$
F_X(x)
=
1-e^{-x}(1+x),
\qquad
x>0.
$$

Therefore

$$
P(X\le2)
=
1-3e^{-2}
\approx0.5940.
$$


In [ ]:
gamma_x = widgets.FloatSlider(value=2.0, min=0.1, max=8.0, step=0.1, description="x")
gamma_cdf_output = widgets.Output()


def update_gamma_cdf_example(*_):
    with gamma_cdf_output:
        clear_output(wait=True)

        x = gamma_x.value
        exact = 1 - math.exp(-x)*(1+x)
        numeric = gamma_cdf_numerical(x, 2, 1)

        display(Math(r"F_X(x)=" + f"{exact:.8f}"))
        display(Math(r"\text{numerical integral}\approx" + f"{numeric:.8f}"))


gamma_x.observe(update_gamma_cdf_example, names="value")
display(widgets.VBox([gamma_x, gamma_cdf_output]))
update_gamma_cdf_example()


### Gamma moments

For every positive integer $k$,

$$
\boxed{
\mathbb E[X^k]
=
\frac{\Gamma(\alpha+k)}
{\lambda^k\Gamma(\alpha)}.
}
$$

In particular,

$$
\boxed{
\mathbb E[X]
=
\frac{\alpha}{\lambda},
\qquad
\operatorname{Var}(X)
=
\frac{\alpha}{\lambda^2}.
}
$$


In [ ]:
gm_alpha = widgets.FloatSlider(value=3.0, min=0.2, max=10.0, step=0.2, description="alpha")
gm_lam = widgets.FloatSlider(value=2.0, min=0.2, max=5.0, step=0.1, description="lambda")
gm_output = widgets.Output()


def update_gamma_moments(*_):
    with gm_output:
        clear_output(wait=True)

        alpha = gm_alpha.value
        lam = gm_lam.value

        mean = alpha/lam
        variance = alpha/(lam*lam)
        second = alpha*(alpha+1)/(lam*lam)

        display(Math(r"\mathbb E[X]=" + f"{mean:.6f}"))
        display(Math(r"\mathbb E[X^2]=" + f"{second:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))


for control in (gm_alpha, gm_lam):
    control.observe(update_gamma_moments, names="value")

display(widgets.VBox([
    widgets.HBox([gm_alpha, gm_lam]),
    gm_output,
]))
update_gamma_moments()


### Exponential as a gamma special case

Since

$$
\Gamma(1)=1,
$$

the gamma density with $\alpha=1$ becomes

$$
\lambda e^{-\lambda x},
\qquad
x>0.
$$

Thus, as probability laws,

$$
\boxed{
\operatorname{Gamma}(1,\lambda)
=
\operatorname{Exp}(\lambda).
}
$$

The displayed density values at the single point $x=0$ are irrelevant to the law.


In [ ]:
x = np.linspace(0.001, 5, 700)

fig, ax = plt.subplots(figsize=(8, 3.3))
ax.plot(x, gamma_density(x, 1, 2), label="Gamma(1,2)")
ax.plot(x, exp_density(x, 2), linestyle="--", label="Exp(2)")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Gamma(1,lambda) and Exponential(lambda)")
ax.legend()
plt.show()


## 6. Gamma shape and mode

For fixed $\lambda>0$:

- if $0<\alpha<1$, the density decreases from an integrable singularity at zero;
- if $\alpha=1$, it is exponential;
- if $\alpha>1$, it increases and then decreases, with unique interior mode

$$
\boxed{
x_{\mathrm{mode}}
=
\frac{\alpha-1}{\lambda}.
}
$$

The mode formula is **not** an interior-mode formula for every $\alpha>0$.


In [ ]:
x = np.linspace(0.03, 7, 1000)

fig, ax = plt.subplots(figsize=(8, 3.6))
for alpha in [0.5, 1.0, 3.0]:
    ax.plot(x, gamma_density(x, alpha, 1), label=f"alpha={alpha:g}")
ax.set_ylim(0, 1.4)
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Gamma densities at common rate lambda=1")
ax.legend()
plt.show()


In [ ]:
shape_alpha = widgets.FloatSlider(value=3.0, min=0.3, max=8.0, step=0.1, description="alpha")
shape_lam = widgets.FloatSlider(value=2.0, min=0.2, max=5.0, step=0.1, description="lambda")
shape_output = widgets.Output()


def update_gamma_shape(*_):
    with shape_output:
        clear_output(wait=True)

        alpha = shape_alpha.value
        lam = shape_lam.value

        mean = alpha/lam

        if alpha > 1:
            mode = (alpha-1)/lam
            display(Math(r"x_{\mathrm{mode}}=" + f"{mode:.6f}"))
        elif abs(alpha-1) < 1e-10:
            display(Markdown("**Exponential case:** strictly decreasing on $(0,\\infty)$."))
        else:
            display(Markdown("**Endpoint singularity:** strictly decreasing on $(0,\\infty)$."))

        display(Math(r"\mathbb E[X]=" + f"{mean:.6f}"))

        upper = max(8/lam, mean + 5*math.sqrt(alpha)/lam)
        x = np.linspace(0.02, upper, 900)

        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.plot(x, gamma_density(x, alpha, lam))
        ax.set_xlabel("x")
        ax.set_ylabel("density")
        ax.set_title("Interactive gamma shape--rate density")
        plt.show()


for control in (shape_alpha, shape_lam):
    control.observe(update_gamma_shape, names="value")

display(widgets.VBox([
    widgets.HBox([shape_alpha, shape_lam]),
    shape_output,
]))
update_gamma_shape()


### Integer shapes and Erlang laws

When $\alpha$ is a positive integer, the gamma law is often called an **Erlang distribution**.

Later, in the Poisson-process chapter, the integer-shape gamma law appears as the waiting time until a specified arrival.


## 7. The Gaussian integral

Before defining the standard normal law, we need

$$
\boxed{
\int_{-\infty}^{\infty}
e^{-x^2}\,dx
=
\sqrt\pi.
}
$$

The proof is one-dimensional in its conclusion but genuinely two-dimensional in its main step.


### Square--disk squeeze

Define

$$
I_R
=
\int_{-R}^{R}
e^{-x^2}\,dx.
$$

Then

$$
I_R^2
=
\iint_{[-R,R]^2}
e^{-(x^2+y^2)}\,dx\,dy.
$$

The disk of radius $R$ lies inside the square, while the square lies inside the disk of radius $\sqrt2R$.

Polar coordinates give

$$
\iint_{x^2+y^2\le a^2}
e^{-(x^2+y^2)}\,dx\,dy
=
\pi(1-e^{-a^2}).
$$

Hence

$$
\pi(1-e^{-R^2})
\le
I_R^2
\le
\pi(1-e^{-2R^2}).
$$

Letting $R\to\infty$ yields $I_R^2\to\pi$ and therefore $I_R\to\sqrt\pi$.


In [ ]:
R_widget = widgets.FloatSlider(value=1.5, min=0.2, max=4.0, step=0.1, description="R")
gauss_geom_output = widgets.Output()


def update_gaussian_geometry(*_):
    with gauss_geom_output:
        clear_output(wait=True)

        R = R_widget.value

        theta = np.linspace(0, 2*math.pi, 600)

        fig, ax = plt.subplots(figsize=(5, 5))
        ax.plot(R*np.cos(theta), R*np.sin(theta), label="radius R disk")
        ax.plot(
            math.sqrt(2)*R*np.cos(theta),
            math.sqrt(2)*R*np.sin(theta),
            linestyle="--",
            label="radius sqrt(2)R disk",
        )
        ax.plot(
            [-R,R,R,-R,-R],
            [-R,-R,R,R,-R],
            label="square",
        )
        ax.set_aspect("equal")
        ax.legend()
        ax.set_title("Disk--square--disk squeeze")
        plt.show()

        lower = math.pi*(1-math.exp(-R*R))
        upper = math.pi*(1-math.exp(-2*R*R))

        grid = np.linspace(-R, R, 10000)
        IR = numerical_integral(np.exp(-grid*grid), grid)

        display(Math(r"\text{lower bound}=" + f"{lower:.8f}"))
        display(Math(r"I_R^2=" + f"{IR*IR:.8f}"))
        display(Math(r"\text{upper bound}=" + f"{upper:.8f}"))


R_widget.observe(update_gaussian_geometry, names="value")
display(widgets.VBox([R_widget, gauss_geom_output]))
update_gaussian_geometry()


### Gamma connection

With $x=t^2$,

$$
\Gamma\left(\frac12\right)
=
2\int_0^\infty e^{-t^2}\,dt
=
\sqrt\pi.
$$

Thus the gamma and normal families are linked through the Gaussian integral.


## 8. Standard normal distribution

A random variable $Z$ is standard normal when

$$
Z\sim N(0,1)
$$

and has density

$$
\boxed{
\phi(z)
=
\frac1{\sqrt{2\pi}}
e^{-z^2/2}.
}
$$

Its cdf is

$$
\boxed{
\Phi(z)
=
\int_{-\infty}^{z}
\phi(t)\,dt.
}
$$


In [ ]:
z = np.linspace(-4, 4, 1000)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(z, phi(z))
ax.set_xlabel("z")
ax.set_ylabel("phi(z)")
ax.set_title("Standard normal density")
plt.show()


### Symmetry

The density is even:

$$
\phi(-z)=\phi(z).
$$

Therefore

$$
\boxed{
\Phi(-z)
=
1-\Phi(z).
}
$$

Equivalently,

$$
P(Z>z)
=
P(Z<-z).
$$


In [ ]:
sym_z = widgets.FloatSlider(value=1.5, min=0.0, max=4.0, step=0.1, description="z")
sym_output = widgets.Output()


def update_normal_symmetry(*_):
    with sym_output:
        clear_output(wait=True)

        z = sym_z.value
        left = Phi_scalar(-z)
        right = 1-Phi_scalar(z)

        display(Math(r"\Phi(-z)=" + f"{left:.8f}"))
        display(Math(r"1-\Phi(z)=" + f"{right:.8f}"))


sym_z.observe(update_normal_symmetry, names="value")
display(widgets.VBox([sym_z, sym_output]))
update_normal_symmetry()


### Mean and variance

For $Z\sim N(0,1)$,

$$
\boxed{
\mathbb E[Z]=0,
\qquad
\operatorname{Var}(Z)=1.
}
$$

The mean follows from symmetry after verifying integrability, and the second moment follows by integration by parts using

$$
\phi'(z)=-z\phi(z).
$$


In [ ]:
grid = np.linspace(-8, 8, 60000)
density = phi(grid)

mean = numerical_integral(grid*density, grid)
second = numerical_integral((grid**2)*density, grid)

display(Math(r"\text{numerical mean}\approx" + f"{mean:.10f}"))
display(Math(r"\text{numerical second moment}\approx" + f"{second:.10f}"))


### No elementary antiderivative

The function

$$
e^{-z^2/2}
$$

has no elementary antiderivative.

Normal probabilities are therefore evaluated numerically, from tables or with software. This is a computational issue, not a defect in the definition of $\Phi$.


## 9. General normal distribution

For $\mu\in\mathbb R$ and $\sigma>0$,

$$
X\sim N(\mu,\sigma^2)
$$

means

$$
\boxed{
f_X(x)
=
\frac1{\sigma\sqrt{2\pi}}
\exp\left(
-\frac{(x-\mu)^2}{2\sigma^2}
\right).
}
$$

The notation uses $\sigma^2$ as the second displayed parameter because it is the variance.


### Normal cdf through the standard normal cdf

For every $x$,

$$
\boxed{
F_X(x)
=
\Phi\left(
\frac{x-\mu}{\sigma}
\right).
}
$$


### Standardization

If

$$
X\sim N(\mu,\sigma^2),
$$

then

$$
\boxed{
Z
=
\frac{X-\mu}{\sigma}
\sim N(0,1).
}
$$

Conversely, if $Z\sim N(0,1)$, then

$$
X=\mu+\sigma Z
$$

has distribution $N(\mu,\sigma^2)$.


In [ ]:
std_mu = widgets.FloatSlider(value=10, min=-10, max=20, step=0.5, description="mu")
std_sigma = widgets.FloatSlider(value=2, min=0.2, max=6, step=0.2, description="sigma")
std_x = widgets.FloatSlider(value=13, min=-10, max=30, step=0.5, description="x")
std_output = widgets.Output()


def update_standardization(*_):
    with std_output:
        clear_output(wait=True)

        mu = std_mu.value
        sigma = std_sigma.value
        x = std_x.value

        z = (x-mu)/sigma
        cdf_value = Phi_scalar(z)

        display(Math(r"z=\frac{x-\mu}{\sigma}=" + f"{z:.6f}"))
        display(Math(r"P(X\le x)=\Phi(z)=" + f"{cdf_value:.8f}"))


for control in (std_mu, std_sigma, std_x):
    control.observe(update_standardization, names="value")

display(widgets.VBox([
    widgets.HBox([std_mu, std_sigma, std_x]),
    std_output,
]))
update_standardization()


### Mean, variance and standard deviation

If

$$
X\sim N(\mu,\sigma^2),
$$

then

$$
\boxed{
\mathbb E[X]=\mu,
\qquad
\operatorname{Var}(X)=\sigma^2.
}
$$

Therefore

$$
\operatorname{SD}(X)=\sigma.
$$

For example, if

$$
X\sim N(10,4),
$$

then

$$
\mathbb E[X]=10,
\qquad
\operatorname{Var}(X)=4,
\qquad
\operatorname{SD}(X)=2.
$$


### Interval probabilities

For $a<b$,

$$
\boxed{
P(a<X\le b)
=
\Phi\left(
\frac{b-\mu}{\sigma}
\right)
-
\Phi\left(
\frac{a-\mu}{\sigma}
\right).
}
$$

Normal laws have densities, so endpoint choices do not affect the probability.


In [ ]:
norm_mu = widgets.FloatSlider(value=10, min=-10, max=20, step=0.5, description="mu")
norm_sigma = widgets.FloatSlider(value=2, min=0.2, max=6, step=0.2, description="sigma")
norm_a = widgets.FloatSlider(value=8, min=-10, max=20, step=0.5, description="a")
norm_b = widgets.FloatSlider(value=13, min=-5, max=30, step=0.5, description="b")
norm_interval_output = widgets.Output()


def update_normal_interval(*_):
    with norm_interval_output:
        clear_output(wait=True)

        mu = norm_mu.value
        sigma = norm_sigma.value
        a = norm_a.value
        b = norm_b.value

        if not a < b:
            display(Markdown("**Require a<b.**"))
            return

        za = (a-mu)/sigma
        zb = (b-mu)/sigma
        probability = Phi_scalar(zb)-Phi_scalar(za)

        display(Math(r"z_a=" + f"{za:.6f}"))
        display(Math(r"z_b=" + f"{zb:.6f}"))
        display(Math(r"P(a<X\le b)=" + f"{probability:.8f}"))


for control in (norm_mu, norm_sigma, norm_a, norm_b):
    control.observe(update_normal_interval, names="value")

display(widgets.VBox([
    widgets.HBox([norm_mu, norm_sigma]),
    widgets.HBox([norm_a, norm_b]),
    norm_interval_output,
]))
update_normal_interval()


## 10. Parameter effects in the normal family

Changing $\mu$ shifts the curve horizontally without changing its shape.

Changing $\sigma$ changes the horizontal scale: larger $\sigma$ spreads probability more widely and lowers the peak.

This is why $\mu$ is a location parameter and $\sigma$ is a scale parameter.


In [ ]:
x = np.linspace(-7, 7, 1000)

fig, ax = plt.subplots(figsize=(8, 3.4))
for mu in [-2, 0, 2]:
    ax.plot(x, normal_density(x, mu, 1), label=f"mu={mu}")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Changing the normal location parameter with sigma=1")
ax.legend()
plt.show()


In [ ]:
x = np.linspace(-7, 7, 1000)

fig, ax = plt.subplots(figsize=(8, 3.4))
for sigma in [0.7, 1.0, 2.0]:
    ax.plot(x, normal_density(x, 0, sigma), label=f"sigma={sigma:g}")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Changing the normal scale parameter with mu=0")
ax.legend()
plt.show()


### Frequently used central probabilities

For $Z\sim N(0,1)$,

$$
P(|Z|\le c)
=
2\Phi(c)-1.
$$

Common numerical values are approximately

$$
P(|Z|\le1)\approx0.6827,
$$

$$
P(|Z|\le2)\approx0.9545,
$$

$$
P(|Z|\le3)\approx0.9973.
$$

These are numerical consequences of $\Phi$, not separate axioms.


In [ ]:
for c in [1,2,3]:
    p = 2*Phi_scalar(c)-1
    display(Math(
        r"P(|Z|\le" + str(c) + r")=" + f"{p:.6f}"
    ))


## 11. Comparing the three families

| Family | Support | Parameters | Mean | Variance |
|---|---|---|---:|---:|
| Exponential | $[0,\infty)$ | rate $\lambda>0$ | $1/\lambda$ | $1/\lambda^2$ |
| Gamma | $[0,\infty)$ | shape $\alpha>0$, rate $\lambda>0$ | $\alpha/\lambda$ | $\alpha/\lambda^2$ |
| Normal | $\mathbb R$ | location $\mu$, scale $\sigma>0$ | $\mu$ | $\sigma^2$ |

A proposed model should first be checked against the support of the quantity being modeled.


### Same mean does not imply same law

The variables

$$
X\sim\operatorname{Exp}(1),
$$

$$
Y\sim\operatorname{Gamma}(2,2),
$$

and

$$
Z\sim N(1,1)
$$

all have mean $1$.

Their variances are

$$
1,
\qquad
\frac12,
\qquad
1.
$$

In particular, $X$ and $Z$ have the same mean and variance but different laws, supports and shapes.


In [ ]:
# Representative at-a-glance sketches, matching the chapter's visual summary.

x_pos = np.linspace(0, 6, 800)

fig, ax = plt.subplots(figsize=(8, 3.0))
ax.plot(x_pos, exp_density(x_pos, 1))
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("At-a-glance: Exp(1)")
plt.show()

x_gamma = np.linspace(0.01, 8, 800)

fig, ax = plt.subplots(figsize=(8, 3.0))
ax.plot(x_gamma, gamma_density(x_gamma, 3, 1))
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("At-a-glance: Gamma(3,1)")
plt.show()

x_norm = np.linspace(-4, 4, 800)

fig, ax = plt.subplots(figsize=(8, 3.0))
ax.plot(x_norm, phi(x_norm))
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("At-a-glance: N(0,1)")
plt.show()


## 12. Historical problem: de Moivre and the normal approximation

For

$$
X\sim\operatorname{Bin}(n,p),
$$

the exact probability of an interval can require a long binomial sum.

The binomial mean and standard deviation are

$$
\mu=np,
\qquad
\sigma=\sqrt{np(1-p)}.
$$

For large $n$, the standardized binomial law is close to $N(0,1)$.

The rigorous limiting theorem belongs to the later chapter on limit theorems.


### Continuity correction

For integers $a\le b$,

$$
\boxed{
P(a\le X\le b)
\approx
\Phi\left(
\frac{b+1/2-np}
{\sqrt{np(1-p)}}
\right)
-
\Phi\left(
\frac{a-1/2-np}
{\sqrt{np(1-p)}}
\right).
}
$$

The half-unit correction reflects the fact that the discrete mass at integer $k$ is represented by the continuous interval from $k-1/2$ to $k+1/2$.

This is still an approximation, not an identity.


### One hundred fair tosses

Let

$$
X\sim\operatorname{Bin}(100,1/2).
$$

For

$$
P(40\le X\le60),
$$

we have

$$
\mu=50,
\qquad
\sigma=5.
$$

Continuity correction gives

$$
P(40\le X\le60)
\approx
P(-2.1<Z<2.1)
=
2\Phi(2.1)-1.
$$


In [ ]:
n = 100
p = 0.5
a = 40
b = 60

exact = binomial_interval_probability(a,b,n,p)
approx = normal_approx_binomial_interval(a,b,n,p,continuity=True)
uncorrected = normal_approx_binomial_interval(a,b,n,p,continuity=False)

display(Math(r"\text{exact binomial probability}=" + f"{exact:.8f}"))
display(Math(r"\text{continuity-corrected normal approximation}=" + f"{approx:.8f}"))
display(Math(r"\text{uncorrected normal approximation}=" + f"{uncorrected:.8f}"))
display(Math(r"\text{corrected absolute error}=" + f"{abs(exact-approx):.8f}"))


In [ ]:
approx_n = widgets.IntSlider(value=100, min=20, max=500, step=10, description="n")
approx_p = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description="p")
approx_width = widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.1, description="sd width")
approx_output = widgets.Output()


def update_normal_approx(*_):
    with approx_output:
        clear_output(wait=True)

        n = approx_n.value
        p = approx_p.value
        width = approx_width.value

        mu = n*p
        sigma = math.sqrt(n*p*(1-p))

        a = max(0, math.ceil(mu-width*sigma))
        b = min(n, math.floor(mu+width*sigma))

        exact = binomial_interval_probability(a,b,n,p)
        corrected = normal_approx_binomial_interval(a,b,n,p,True)
        uncorrected = normal_approx_binomial_interval(a,b,n,p,False)

        display(Markdown(f"**Integer interval:** [{a},{b}]"))
        display(Math(r"\text{exact}=" + f"{exact:.8f}"))
        display(Math(r"\text{corrected approximation}=" + f"{corrected:.8f}"))
        display(Math(r"\text{uncorrected approximation}=" + f"{uncorrected:.8f}"))
        display(Math(r"\text{corrected error}=" + f"{abs(exact-corrected):.8f}"))


for control in (approx_n, approx_p, approx_width):
    control.observe(update_normal_approx, names="value")

display(widgets.VBox([
    widgets.HBox([approx_n, approx_p, approx_width]),
    approx_output,
]))
update_normal_approx()


### Numerical experiment across sample sizes

When comparing different $n$, it is misleading to keep a fixed raw interval such as $[40,60]$.

A fairer comparison keeps the standardized width approximately constant by choosing endpoints near

$$
np\pm2\sqrt{np(1-p)}.
$$

The experiment below is numerical evidence about approximation quality. It is not a proof of convergence.


In [ ]:
sample_sizes = [20, 50, 100, 200, 500, 1000]
p = 0.5

errors = []

for n in sample_sizes:
    mu = n*p
    sigma = math.sqrt(n*p*(1-p))

    a = max(0, math.ceil(mu-2*sigma))
    b = min(n, math.floor(mu+2*sigma))

    exact = binomial_interval_probability(a,b,n,p)
    approx = normal_approx_binomial_interval(a,b,n,p,True)
    errors.append(abs(exact-approx))

    display(Markdown(
        f"n={n}: interval=[{a},{b}], exact={exact:.7f}, "
        f"approx={approx:.7f}, error={abs(exact-approx):.7g}"
    ))

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(sample_sizes, errors, marker="o")
ax.set_xlabel("n")
ax.set_ylabel("absolute error")
ax.set_title("Numerical error of continuity-corrected normal approximation")
plt.show()


## 13. Simulation laboratory: three continuous families

The next experiment simulates from:

$$
\operatorname{Exp}(2),
$$

$$
\operatorname{Gamma}(3,2),
$$

and

$$
N(1,4),
$$

and compares sample means and variances with theoretical values.

The simulation checks implementation and illustrates sampling variability. It does not replace the analytic derivations.


In [ ]:
sim_N = widgets.IntSlider(value=20000, min=1000, max=100000, step=1000, description="N")
sim_seed = widgets.IntSlider(value=2026, min=0, max=5000, description="seed")
sim_output = widgets.Output()


def update_family_sim(*_):
    with sim_output:
        clear_output(wait=True)

        N = sim_N.value
        seed = sim_seed.value
        rng = np.random.default_rng(seed)

        samples = {
            "Exp(2)": (
                rng.exponential(scale=1/2, size=N),
                1/2,
                1/4,
            ),
            "Gamma(3,2)": (
                rng.gamma(shape=3, scale=1/2, size=N),
                3/2,
                3/4,
            ),
            "N(1,4)": (
                rng.normal(loc=1, scale=2, size=N),
                1,
                4,
            ),
        }

        rows = [
            "| family | sample mean | theory mean | sample variance | theory variance |",
            "|---|---:|---:|---:|---:|",
        ]

        for name, (sample, theory_mean, theory_var) in samples.items():
            rows.append(
                f"| {name} | {sample.mean():.5f} | {theory_mean:.5f} | "
                f"{sample.var():.5f} | {theory_var:.5f} |"
            )

        display(Markdown("\n".join(rows)))


for control in (sim_N, sim_seed):
    control.observe(update_family_sim, names="value")

display(widgets.VBox([
    widgets.HBox([sim_N, sim_seed]),
    sim_output,
]))
update_family_sim()


## 14. Solved-style computational checks


### Exponential tail and interval probabilities

If

$$
X\sim\operatorname{Exp}(3),
$$

then

$$
P(X>1)=e^{-3},
$$

$$
P(X\le1/2)=1-e^{-3/2},
$$

and

$$
P(1/2<X\le1)
=
e^{-3/2}-e^{-3}.
$$


In [ ]:
display(Math(r"P(X>1)=" + f"{math.exp(-3):.8f}"))
display(Math(r"P(X\le1/2)=" + f"{1-math.exp(-1.5):.8f}"))
display(Math(r"P(1/2<X\le1)=" + f"{math.exp(-1.5)-math.exp(-3):.8f}"))


### Recovering a gamma model from mean and variance

If a gamma variable has

$$
\mathbb E[X]=10
$$

and

$$
\operatorname{Var}(X)=20,
$$

then

$$
\frac{\alpha}{\lambda}=10,
$$

$$
\frac{\alpha}{\lambda^2}=20.
$$

Dividing gives

$$
\lambda=\frac12,
$$

and therefore

$$
\alpha=5.
$$


In [ ]:
m = 10
v = 20

lam = m/v
alpha = m*m/v

display(Math(r"\lambda=" + f"{lam:g}"))
display(Math(r"\alpha=" + f"{alpha:g}"))


### Gamma scaling

If

$$
X\sim\operatorname{Gamma}(\alpha,\lambda)
$$

and

$$
Y=\lambda X,
$$

then $Y$ has density

$$
f_Y(y)
=
\frac1{\Gamma(\alpha)}
y^{\alpha-1}e^{-y},
\qquad
y>0.
$$

Thus the rate can be removed by a deterministic scaling.


### Standard normal interval

If $Z\sim N(0,1)$,

$$
P(-1.5<Z<0.7)
=
\Phi(0.7)-\Phi(-1.5).
$$

Also,

$$
P(Z>2)
=
1-\Phi(2),
$$

and

$$
P(|Z|>c)
=
2(1-\Phi(c)),
\qquad
c\ge0.
$$


In [ ]:
display(Math(
    r"P(-1.5<Z<0.7)="
    + f"{Phi_scalar(0.7)-Phi_scalar(-1.5):.8f}"
))
display(Math(
    r"P(Z>2)="
    + f"{1-Phi_scalar(2):.8f}"
))


### Symmetric normal interval

If

$$
X\sim N(12,25),
$$

then $\sigma=5$, and

$$
P(7<X<17)
=
P(-1<Z<1)
=
2\Phi(1)-1.
$$


In [ ]:
display(Math(
    r"P(7<X<17)="
    + f"{2*Phi_scalar(1)-1:.8f}"
))


## 15. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Exponential", "exp"),
        ("Memoryless", "memoryless"),
        ("Gamma function", "gammafun"),
        ("Gamma law", "gamma"),
        ("Gamma mode", "gammamode"),
        ("Standard normal", "standard"),
        ("General normal", "normal"),
        ("Normal approximation", "approx"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "exp",
            "memoryless",
            "gammafun",
            "gamma",
            "gammamode",
            "standard",
            "normal",
            "approx",
        ])

    if kind == "exp":
        target = "0.2"
        prompt = "An Exp(lambda) variable has mean 5. Find lambda."
        hint = "Use E[X]=1/lambda."
        solution = r"\lambda=1/5=0.2."

    elif kind == "memoryless":
        target = str(math.exp(-0.6))
        prompt = "X~Exp(0.2). Given X>5, find P(X>8|X>5) as a decimal."
        hint = "Memorylessness reduces the extra wait to 3."
        solution = r"P(X>8\mid X>5)=e^{-0.6}."

    elif kind == "gammafun":
        target = "6"
        prompt = "Find Gamma(4)."
        hint = "Gamma(n)=(n-1)!."
        solution = r"\Gamma(4)=3!=6."

    elif kind == "gamma":
        target = "0.75"
        prompt = "X~Gamma(3,2) in shape-rate form. Find Var(X)."
        hint = "Use alpha/lambda^2."
        solution = r"\operatorname{Var}(X)=3/4."

    elif kind == "gammamode":
        target = "1"
        prompt = "X~Gamma(3,2). Find the interior mode."
        hint = "For alpha>1, use (alpha-1)/lambda."
        solution = r"x_{\mathrm{mode}}=1."

    elif kind == "standard":
        target = "0"
        prompt = "For Z~N(0,1), find E[Z]."
        hint = "Use symmetry after integrability."
        solution = r"\mathbb E[Z]=0."

    elif kind == "normal":
        target = "2"
        prompt = "X~N(10,4). Find the standard deviation."
        hint = "The second displayed parameter is the variance."
        solution = r"\operatorname{SD}(X)=2."

    else:
        target = "no"
        prompt = "Does a continuity correction make the normal approximation exactly equal to a binomial probability? yes/no"
        hint = "It remains an approximation."
        solution = r"\text{No.}"

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )
    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))
    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ", "")
        target = state["target"].replace(" ", "")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Check the parameterization and structural formula first.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind, new_button]),
    prompt_output,
    widgets.HBox([answer_box, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    feedback_output,
]))

make_exercise()


## 16. AI Audit: exponential, gamma and normal claims

Audit any AI-generated solution using the following checklist:

1. Is $\lambda$ treated as a **rate** in the exponential law?
2. Is the exponential survival function written as $e^{-\lambda x}$?
3. Is memorylessness stated as an exact conditional-probability identity?
4. Is memorylessness incorrectly treated as a generic property of continuous laws?
5. Is the positivity assumption in the exponential characterization acknowledged?
6. Does inverse-transform simulation use the correct logarithmic formula?
7. Is $\Gamma(n)$ correctly identified as $(n-1)!$, not $n!$?
8. Is the gamma law using the chapter's **shape--rate** parameterization?
9. Is the distinction between rate and scale made explicit?
10. For $0<\alpha<1$, is the endpoint singularity handled correctly?
11. Is the mode formula $(\alpha-1)/\lambda$ used only for $\alpha>1$ as an interior mode?
12. Is $\operatorname{Gamma}(1,\lambda)$ recognized as the exponential law?
13. Is the Gaussian integral normalized before the standard normal density is used?
14. Is $\Gamma(1/2)=\sqrt\pi$ correctly connected to the Gaussian integral?
15. Is $\Phi(-z)=1-\Phi(z)$ used correctly?
16. Is the standard deviation of $N(\mu,\sigma^2)$ correctly identified as $\sigma$, not $\sigma^2$?
17. Is standardization performed with $(X-\mu)/\sigma$?
18. Are normal interval probabilities converted to $\Phi$ correctly?
19. Are the roles of $\mu$ and $\sigma$ distinguished as location and scale?
20. Is support compatibility checked before choosing a model?
21. Is the de Moivre--Laplace approximation clearly labeled as an approximation?
22. Is the half-unit continuity correction applied in the correct direction?
23. Is a few numerical comparisons being incorrectly presented as a proof of a limit theorem?

### Claims to audit

- “For every $\alpha>0$, the mode of $\operatorname{Gamma}(\alpha,\lambda)$ is $(\alpha-1)/\lambda$.”
- “The exponential memoryless property is only approximately true for long waits.”
- “In $N(\mu,\sigma^2)$, the quantity $\sigma^2$ is the standard deviation.”
- “A continuity-corrected normal approximation is an exact binomial identity.”

All four statements are false as written.


### Suggested AI-guided activities

- “Start with an exponential mean and make me recover the rate before computing a survival probability.”
- “Give me a gamma mean and variance and require me to identify shape and rate.”
- “Make me classify the gamma density for $\alpha<1$, $\alpha=1$, and $\alpha>1$ before computing a mode.”
- “Guide me through the Gaussian-integral square--disk squeeze without revealing the final answer immediately.”
- “Give me a general normal probability and make me standardize it before using $\Phi$.”
- “Give me a binomial interval, require the continuity correction, and compare exact and approximate values numerically while explicitly separating experiment from proof.”


## 17. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. For X~Exp(lambda), E[X] equals:",
        ["Choose...", "lambda", "1/lambda", "1/lambda^2"],
        "1/lambda",
        r"\mathbb E[X]=1/\lambda.",
    ),
    (
        "2. The exponential memoryless identity is exact:",
        ["Choose...", "true", "false"],
        "true",
        r"P(X>s+t\mid X>s)=P(X>t).",
    ),
    (
        "3. Gamma(n) for positive integer n equals:",
        ["Choose...", "n!", "(n-1)!", "1/n!"],
        "(n-1)!",
        r"\Gamma(n)=(n-1)!.",
    ),
    (
        "4. This book parameterizes Gamma(alpha,lambda) using:",
        ["Choose...", "rate lambda", "scale lambda"],
        "rate lambda",
        r"\mathbb E[X]=\alpha/\lambda.",
    ),
    (
        "5. For 0<alpha<1, the gamma density:",
        ["Choose...", "has an interior mode", "decreases from an endpoint singularity"],
        "decreases from an endpoint singularity",
        r"\text{The interior-mode formula applies only when }\alpha>1.",
    ),
    (
        "6. Gamma(1,lambda) is:",
        ["Choose...", "Exp(lambda)", "N(0,1)", "U(0,lambda)"],
        "Exp(lambda)",
        r"\operatorname{Gamma}(1,\lambda)=\operatorname{Exp}(\lambda).",
    ),
    (
        "7. Gamma(1/2) equals:",
        ["Choose...", "sqrt(pi)", "pi", "1/sqrt(pi)"],
        "sqrt(pi)",
        r"\Gamma(1/2)=\sqrt\pi.",
    ),
    (
        "8. For Z~N(0,1), Phi(-z) equals:",
        ["Choose...", "1-Phi(z)", "Phi(z)", "-Phi(z)"],
        "1-Phi(z)",
        r"\Phi(-z)=1-\Phi(z).",
    ),
    (
        "9. For X~N(mu,sigma^2), standard deviation is:",
        ["Choose...", "sigma", "sigma^2", "mu"],
        "sigma",
        r"\operatorname{SD}(X)=\sigma.",
    ),
    (
        "10. Standardization uses:",
        ["Choose...", "(X-mu)/sigma", "(X-mu)/sigma^2", "X/sigma"],
        "(X-mu)/sigma",
        r"Z=(X-\mu)/\sigma.",
    ),
    (
        "11. Increasing sigma while mu is fixed:",
        ["Choose...", "spreads the curve", "only shifts the curve", "raises the peak"],
        "spreads the curve",
        r"\text{Larger }\sigma\text{ spreads the same total mass more widely.}",
    ),
    (
        "12. Continuity correction makes a normal approximation exact:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{It often improves the approximation but does not create an identity.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="470px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:650px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_, _, correct, _) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i, (
            widget,
            (_, _, correct, explanation),
        ) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows + [grade_button, quiz_output]
))


## 18. Automatic mathematical verification

The final cell checks representative formulas from all three families and the normal approximation.


In [ ]:
# Exponential normalization, mean, variance, memorylessness.
for lam in [0.5, 1.0, 2.0, 4.0]:
    grid = np.linspace(0, 40/lam, 50000)
    mass = numerical_integral(exp_density(grid, lam), grid)
    assert abs(mass-1) < 2e-7

    s, t = 2.3, 1.7
    lhs = math.exp(-lam*(s+t))/math.exp(-lam*s)
    rhs = math.exp(-lam*t)
    assert abs(lhs-rhs) < 1e-12

# Gamma recurrence.
for n in range(1, 9):
    assert abs(math.gamma(n)-math.factorial(n-1)) < 1e-12

# Gamma special case.
x = np.linspace(0.001, 6, 1000)
assert np.allclose(
    gamma_density(x, 1, 2),
    exp_density(x, 2),
    atol=1e-12,
)

# Gamma moments.
alpha, lam = 3, 2
assert alpha/lam == 1.5
assert alpha/(lam*lam) == 0.75
assert (alpha-1)/lam == 1

# Gamma(2,1) cdf.
x0 = 2
exact_gamma_cdf = 1-math.exp(-x0)*(1+x0)
assert abs(exact_gamma_cdf-(1-3*math.exp(-2))) < 1e-12

# Gaussian integral numerical check.
grid = np.linspace(-8, 8, 100000)
gauss = numerical_integral(np.exp(-grid*grid), grid)
assert abs(gauss-math.sqrt(math.pi)) < 1e-8

# Gamma(1/2).
assert abs(math.gamma(0.5)-math.sqrt(math.pi)) < 1e-12

# Standard normal normalization, moments and symmetry.
grid = np.linspace(-8, 8, 100000)
dens = phi(grid)

mass = numerical_integral(dens, grid)
mean = numerical_integral(grid*dens, grid)
second = numerical_integral((grid**2)*dens, grid)

assert abs(mass-1) < 1e-12
assert abs(mean) < 1e-12
assert abs(second-1) < 1e-12

for z in [0.2, 1, 1.5, 2.3]:
    assert abs(Phi_scalar(-z) - (1-Phi_scalar(z))) < 1e-15

# General normal.
mu, sigma = 10, 2
assert abs(float(normal_cdf(np.array([13.0]),mu,sigma)[0])-Phi_scalar(1.5)) < 1e-15

# Central standard-normal probabilities.
assert abs((2*Phi_scalar(1)-1)-0.682689492) < 1e-8
assert abs((2*Phi_scalar(2)-1)-0.954499736) < 1e-8

# One hundred fair tosses.
exact = binomial_interval_probability(40,60,100,0.5)
approx = normal_approx_binomial_interval(40,60,100,0.5,True)

assert abs(exact-0.9647997998) < 1e-9
assert abs(approx-0.9642711589) < 1e-9
assert abs(exact-approx) < 0.001

show_result(
    "All Chapter 10 automatic checks passed",
    r"P(X>x)=e^{-\lambda x}",
    r"\mathbb E[\operatorname{Exp}(\lambda)]=1/\lambda",
    r"\Gamma(n)=(n-1)!",
    r"\operatorname{Gamma}(1,\lambda)=\operatorname{Exp}(\lambda)",
    r"\Gamma(1/2)=\sqrt\pi",
    r"\Phi(-z)=1-\Phi(z)",
    r"\frac{X-\mu}{\sigma}\sim N(0,1)",
    note=(
        "Exponential, gamma, Gaussian-normalization, standardization and "
        "de Moivre numerical checks all passed."
    ),
)


## 19. Chapter map

| Chapter concept | Computational representation |
|---|---|
| exponential density | direct normalization |
| rate parameter | density-shape comparison |
| exponential cdf/survival | exact tail and interval widget |
| exponential moments | rate-from-mean calculation |
| memorylessness | conditional-probability identity |
| exponential characterization | linearity of $-\log S(t)$ |
| inverse transformation | uniform-to-exponential simulation |
| gamma function | recurrence and factorial connection |
| gamma shape--rate law | direct density and numerical normalization |
| gamma cdf | incomplete-gamma integral and integer-shape example |
| gamma moments | interactive shape/rate moment calculator |
| exponential as gamma | overlay of $\operatorname{Gamma}(1,\lambda)$ and $\operatorname{Exp}(\lambda)$ |
| gamma shape regimes | $\alpha<1$, $\alpha=1$, $\alpha>1$ density comparison |
| gamma mode | interactive interior-mode calculation |
| Gaussian integral | square--disk squeeze |
| $\Gamma(1/2)$ | Gaussian connection |
| standard normal | $\phi$, $\Phi$, symmetry and moments |
| general normal | cdf through standardization |
| normal interval probabilities | interactive z-score calculator |
| normal location effect | varying $\mu$ |
| normal scale effect | varying $\sigma$ |
| family comparison | support, parameters, mean and variance |
| at-a-glance density sketches | exponential, gamma and normal examples |
| de Moivre--Laplace | exact binomial versus normal approximation |
| continuity correction | corrected and uncorrected comparison |
| simulation laboratory | empirical versus theoretical moments |
| AI Audit | parameterization and approximation checks |

The central message is:

$$
\boxed{
\text{understand the structure of the family first, then compute with its formulas}.
}
$$
